In [ ]:
import os
import torch
import fiftyone
from fiftyone import brain
from fiftyone import zoo 
from fiftyone import ViewField as F
import xml.etree.ElementTree as ET
from datetime import datetime
from xml.dom import minidom
from tqdm import tqdm





In [ ]:
# Check GPU 
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

True
NVIDIA GeForce RTX 4070


In [ ]:
# #NEVER run launch app twice...!!!!!!!!!!!!!!!!

print('Point browser to http://localhost:5159/')
session2 = fiftyone.launch_app(auto = False, port=5159)


Point browser to http://localhost:5159/
Session launched. Run `session.show()` to open the App in a cell output.


In [ ]:
# Paths are HARDCODED , Update accordingly

# #entire dataset. wb + nb
image_path = r"/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/" # Image folder path
base_annotation = r"/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/annotations_uav-mu_neg-set_v4-7.xml" # old images xml
incoming_annotation = r"/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/annotations_new_negatives.xml" # new images xml

# Change thresold accordingly
thresh_value = 0.07

output_path=r"/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/output/"
folder_path = os.path.join(output_path, f"thresh_{thresh_value}")
os.makedirs(folder_path, exist_ok=True)

merged_xml = folder_path + "/old+new.xml"
output_dup_xml = folder_path + f"/duplicates_thrsh-{thresh_value}.xml"
output_dup_filtered_xml = folder_path + f"/Combined_unique _thrsh-{thresh_value}.xml"
filtered_unique_trained_path = folder_path + f"/Unique__old_thrsh-{thresh_value}.xml"  
filtered_unique_incoming_path = folder_path + f"/Unique_new_thrsh-{thresh_value}.xml"    




In [134]:
# Merge base_annotation.xml and incoming_annotation.xml
base_tree = ET.parse(base_annotation)
base_root = base_tree.getroot()

new_tree = ET.parse(incoming_annotation)
new_root = new_tree.getroot()

base_images = base_root.findall("image")
new_images = new_root.findall("image")

# Merge XMLs and Update image IDs
counter = len(base_images)
for image in new_images:
    image.set("id", str(counter))
    base_root.append(image)
    counter += 1


base_tree.write(merged_xml, encoding="utf-8", xml_declaration=True)
print(f"Merged XML saved to: {merged_xml}")

Merged XML saved to: /home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/output/thresh_0.23/annotations_neg_merged.xml


In [ ]:

print(f"Loading data:")
# Import dataset by explicitly providing paths to the source media and labels
name = "CVAT image dataset"
dataset5 = fiftyone.Dataset.from_dir(
    dataset_type=fiftyone.types.CVATImageDataset,
    data_path=image_path,
    labels_path=merged_xml
)



Loading data:
No version tag found; assuming version 1.1
 100% |███████████████| 1906/1906 [351.7ms elapsed, 0s remaining, 5.4K samples/s]      


In [24]:
print(dataset5)

Name:        2026.04.22.08.17.23.951992
Media type:  image
Num samples: 1906
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField


In [ ]:
#The notebook kernal needs to be running in a CUDA activated environment

model = zoo.load_zoo_model("dinov2-vitl14-torch")
print("model loaded")


Using cache found in /home/nithin/.cache/torch/hub/facebookresearch_dinov2_main
/home/nithin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/nithin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/nithin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


model loaded


In [75]:
embeddings = dataset5.compute_embeddings(model, batch_size=64)

Model does not support batching
 100% |███████████████| 1906/1906 [19.8m elapsed, 0s remaining, 2.6 samples/s]      


In [36]:

#embeddings are resued from previous steps
results = brain.compute_uniqueness(dataset5, model = "dinov2-vitl14-torch", embeddings=embeddings)

Computing uniqueness...
Uniqueness computation complete


In [37]:
#embeddings are resued from previous steps
results = brain.compute_similarity(dataset5, model = "dinov2-vitl14-torch", embeddings=embeddings)
print("Done")

Done


In [135]:
## HARDCODED the threshold
results.find_duplicates(thresh=thresh_value)

#Neighbours map also generated
#print(results.neighbors_map)

Computing duplicate samples...
Duplicates computation complete


In [136]:

#duplicates_view helps to visualize the neighbors_map
#It populates the fields: dup_type, dup_id, dup_dist for every samples
duplicates_view = results.duplicates_view(
    type_field="dup_type",
    id_field="dup_id",
    dist_field="dup_dist",
)
print(duplicates_view)
# This view has "nearest" samples and "duplicate" sample
# Each duplicate sample has been found similar to a "nearest" sample
# The "nearest" samples are unique in the dataset


Dataset:     2026.04.22.08.17.23.951992
Media type:  image
Num samples: 1219
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    uniqueness:       fiftyone.core.fields.FloatField
    dup_type:         fiftyone.core.fields.StringField
    dup_id:           fiftyone.core.fields.StringField
    dup_dist:         fiftyone.core.fields.FloatField
View stages:
    1. Select(sample_ids=[np.str_('69e8...90a896fd40b5'), np.str_('69e8...90a896fd4097'), np.str_('69e8...90a896fd40b3'), ...], ordered=True)


In [70]:
# session2.view = duplicates_view#dataset.view()

In [137]:
dups_only = duplicates_view.match(F("dup_type") == "duplicate")
print(len(dups_only))
# print(dups_only)
# Iterate over duplicates only
for sample in dups_only:
    print(sample.filepath)

1046
/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/dd15_flug-06_09-04-2026_11-45-51_004570.png
/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/dd15_flug-06_09-04-2026_11-45-51_006780.png
/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/dd15_flug-06_09-04-2026_11-45-51_004560.png
/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/dd15_flug-06_09-04-2026_11-45-51_004540.png
/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/dd15_flug-06_09-04-2026_11-45-51_003200.png
/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/dd15_flug-06_09-04-2026_11-45-51_004900.png
/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/dd15_flug-06_09-04-2026_11-45-51_007460.png
/home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/negatives_images/dd15_flug-06_09

In [ ]:

# Generating Duplicate images XML
annotations = ET.Element("annotations")


for idx, sample in enumerate(dups_only):
    filepath = sample.filepath
    filename = os.path.basename(filepath)
    image_tag = ET.SubElement(annotations, "image")
    image_tag.set("id", str(idx))
    image_tag.set("name", filename)
    image_tag.set("width", "1024")   
    image_tag.set("height", "768")   

xml_str = ET.tostring(annotations, encoding="utf-8")
parsed = minidom.parseString(xml_str)
pretty_xml = parsed.toprettyxml(indent="  ")  

pretty_xml = "\n".join([line for line in pretty_xml.splitlines() if line.strip()])


with open(output_dup_xml, "w", encoding="utf-8") as f:
    f.write(pretty_xml)

print(f"Duplicate images XML file created: {output_dup_xml}")


Duplicate images XML file created: /home/nithin/Work_Student_Job/dataset/16-04-09_13/negative_dataset/output/thresh_0.23/annotations_neg_all_duplicates_thrsh-0.23.xml


In [ ]:
# Filtering Merged XMl against Duplicate

tree_merged_xml = ET.parse(merged_xml)
root_merged_xml = tree_merged_xml.getroot()

root_ann_filtered = ET.parse(output_dup_xml)
images_filtered = root_ann_filtered.findall('image')

nf = []
for image in images_filtered:
    image_name = str(image.attrib.get('name'))
    filtered_image = root_merged_xml.find(f'image[@name="{image_name}"]')
    filtered_image_jpg = root_merged_xml.find(f'image[@name="{image_name.replace(".png", ".jpg")}"]')
    filtered_image = filtered_image_jpg if filtered_image is None else filtered_image
    if filtered_image is not None:
        root_merged_xml.remove(filtered_image)
    else:
        nf.append(image_name)  

for new_id, image in enumerate(root_merged_xml.findall('image')):
    image.set('id', str(new_id))

tree_merged_xml.write(output_dup_filtered_xml)
print(f"Image not found in original dataset: \n{len(nf)}")

Image not found in original dataset: 
0


In [ ]:

# Filter Base XML and Incoming XML against Duplicate
def get_common_images(original_tree, filter_tree):
    root_original = original_tree.getroot()
    root_filter = filter_tree.getroot()

    original_names = set(img.attrib.get('name') for img in root_original.findall('image'))

    new_root = ET.Element('annotations')

    if root_filter.find('version') is not None:
        ET.SubElement(new_root, 'version').text = root_filter.find('version').text
    if root_filter.find('meta') is not None:
        new_root.append(ET.fromstring(ET.tostring(root_filter.find('meta'))))

    found_count = 0
    not_found = []

    for img in tqdm(root_filter.findall('image'), desc="Filtering images"):
        img_name = img.attrib.get('name')
        if img_name in original_names or img_name.replace(".png", ".jpg") in original_names or img_name.replace(".jpg", ".png") in original_names:
            new_root.append(img)
            found_count += 1
        else:
            not_found.append(img_name)

    for new_id, img in enumerate(new_root.findall('image')):
        img.set('id', str(new_id))

    new_tree = ET.ElementTree(new_root)
    return new_tree, found_count, not_found


tree_A = ET.parse(output_dup_filtered_xml)
tree_B = ET.parse(incoming_annotation)

tree_D, found_count_B, not_found_count_B = get_common_images(tree_A, tree_B)
tree_D.write(filtered_unique_incoming_path)
print(f"{found_count_B} Images kept and {len(not_found_count_B)} Images are removed from incoming data")

tree_C = ET.parse(base_annotation)
tree_E, found_count_C, not_found_count_C = get_common_images(tree_A, tree_C)
tree_E.write(filtered_unique_trained_path)
print(f"{found_count_C}Images kept and {len(not_found_count_C)} Images are removed from trained data")
print(f"Completed....")


Filtering images: 100%|██████████| 1190/1190 [00:00<00:00, 2479494.17it/s]


206 Images kept and 984 Images are removed from incoming data


Filtering images: 100%|██████████| 716/716 [00:00<00:00, 1816770.52it/s]

654Images kept and 62 Images are removed from trained data
Completed....


In [ ]:
session2.close()